# NON-CANONICAL WEBSITE INTERACTIVE NOTEBOOK

## Explore pre-defined gate outcomes (presentation only)

Use `SELECTED_INDEX` in the code cell to pick among **frozen** scenarios that reuse the same case dicts as the archival notebooks. The chart shows **pass/fail bits** for the five gates returned by `evaluate_case`; it does **not** replace the engine and writes **nothing** to `outputs/`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# === NON-CANONICAL WEBSITE INTERACTIVE NOTEBOOK ===

def repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p
    raise RuntimeError("Repository root not found.")


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

# Pre-defined scenarios only (same feature physics the archival notebooks use).
SCENARIOS = [
    {
        "label": "smoke_core_001 (replay reference)",
        "case": {
            "case_id": "smoke_core_001",
            "features": {
                "intrinsic_safety": 0.58,
                "evidence_strength": 0.55,
                "bias_harm_index": 0.44,
                "uncertainty_calibration": 0.52,
                "traceability_integrity": 0.53,
            },
        },
        "mode": eng.MODE_REPLAY,
    },
    {
        "label": "demo_replay_pass (full mode)",
        "case": {
            "case_id": "demo_replay_pass",
            "features": {
                "intrinsic_safety": 0.62,
                "evidence_strength": 0.60,
                "bias_harm_index": 0.40,
                "uncertainty_calibration": 0.58,
                "traceability_integrity": 0.57,
            },
        },
        "mode": eng.MODE_CANONICAL_FULL,
    },
    {
        "label": "demo_gate_fail (full mode)",
        "case": {
            "case_id": "demo_gate_fail",
            "features": {
                "intrinsic_safety": 0.35,
                "evidence_strength": 0.60,
                "bias_harm_index": 0.40,
                "uncertainty_calibration": 0.58,
                "traceability_integrity": 0.57,
            },
        },
        "mode": eng.MODE_CANONICAL_FULL,
    },
    {
        "label": "demo_fullmode_abstention",
        "case": {
            "case_id": "demo_fullmode_abstention",
            "features": {
                "intrinsic_safety": 0.62,
                "evidence_strength": 0.60,
                "bias_harm_index": 0.40,
                "uncertainty_calibration": 0.42,
                "traceability_integrity": 0.50,
                "fallback_safety_delta": 0.15,
            },
        },
        "mode": eng.MODE_CANONICAL_FULL,
    },
]

# Reader adjusts this index (0 .. len(SCENARIOS)-1) and re-runs the cell — no hidden UI.
SELECTED_INDEX = 0

scenario = SCENARIOS[SELECTED_INDEX]
result = eng.evaluate_case(
    scenario["case"], profile_name="moderate", mode=scenario["mode"]
)

gate_cols = [
    ("gate_safety", "Safety"),
    ("gate_evidence", "Evidence"),
    ("gate_bias", "Bias cap"),
    ("gate_calibration", "Calibration"),
    ("gate_traceability", "Traceability"),
]
gate_values = [result[k] for k, _ in gate_cols]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([lbl for _, lbl in gate_cols], gate_values, color="#2c6e49")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Pass (1) / Fail (0)")
ax.set_title(f"{scenario['label']} — moderate profile")

pd.set_option("display.max_columns", None)
summary = pd.Series(
    {
        "governance_outcome": result["governance_outcome"],
        "approved": result["approved"],
        "compensatory_score": result["compensatory_score"],
        "mode": result["mode"],
    }
)
print(summary)
if scenario["mode"] == eng.MODE_CANONICAL_FULL:
    extra = pd.Series(
        {
            "abstention_rate": result.get("abstention_rate"),
            "abstention_triggered": result.get("abstention_triggered"),
            "fallback_adequate": result.get("fallback_adequate"),
        }
    )
    print(extra)

plt.tight_layout()
plt.show()

